## Execution Example
This notebook shows the length-extension forgery against `token_mac`. The tag is `H(secret ‖ data)` with a Merkle-Damgard hash, and the published tag **is** the hash's internal state, so we can keep hashing from it without ever knowing the secret.

In [1]:
from token_mac import forge_token, find_secret_len
from duel1_targets import issue_token, verify_token

In [2]:
token = issue_token()
data: bytes = token["data"].encode()
tag: int = token["tag"]

print(token)

{'data': 'user=alice&role=user', 'tag': 1116660411}


First, the naive attack: simply editing the data and reusing the old tag. The server rejects it, so the MAC does stop a plain edit.

In [3]:
verify_token(b"user=alice&role=admin", tag)

False

The secret length is not given to us, so we forge one token per guess and use the server's own check as an oracle. The forged tag never depends on the secret length; only the glue padding does, so every length with the same value mod 4 is accepted.

In [4]:
extension = b"&role=admin"

accepted = find_secret_len(data, tag, extension)
print("secret lengths accepted by the server:", accepted)
print("secret length mod 4:", {n % 4 for n in accepted})

secret lengths accepted by the server: [1, 5, 9, 13, 17, 21, 25, 29]
secret length mod 4: {1}


In [5]:
forged_data, forged_tag = forge_token(data, tag, accepted[0], extension)

print("forged data:", forged_data)
print("forged tag: ", forged_tag)
print("server accepts forged token:", verify_token(forged_data, forged_tag))
print("contains role=admin:", b"role=admin" in forged_data)

forged data: b'user=alice&role=user\x00\x00\x00&role=admin'
forged tag:  4024164909
server accepts forged token: True
contains role=admin: True


The forged data is `user=alice&role=user` + glue padding (`   `) + `&role=admin`. If the application parses query-style parameters with "last value wins", `alice` is now an admin, and we never learned the secret.

**Answer**: forged token `data = b'user=alice&role=user   &role=admin'` with a tag accepted by `verify_token`, obtained without the secret.